# Phase 4: Understand regime_ml.py - Complete Feature Engineering

Builds 26 production features for regime classification

## Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import os
import pickle
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Connect to DuckDB
con = duckdb.connect('../../../data/investing/investing.duckdb', read_only=True)

# Create output directory
os.makedirs('../outputs', exist_ok=True)

print('Connected to DuckDB warehouse')
print('Building 26 production features for regime classification')


## Part 1A: SPY Momentum & Volatility

Multi-horizon returns and realized volatility from S&P 500 price

In [ ]:
# Load SPY price data
spy_df = con.execute("""
    SELECT date, close FROM silver_macro_prices
    WHERE symbol = 'SPY' ORDER BY date
""").df()

spy_df['date'] = pd.to_datetime(spy_df['date'])
spy = spy_df.set_index('date')['close'].sort_index()

# Initialize feature dataframe
feat = pd.DataFrame(index=spy.index)

# Multi-horizon returns (momentum at different timescales)
feat['ret_spy_1m'] = spy.pct_change(21)      # Past 1 month (21 trading days)
feat['ret_spy_3m'] = spy.pct_change(63)      # Past 3 months (63 trading days)
feat['ret_spy_6m'] = spy.pct_change(126)     # Past 6 months
feat['ret_spy_12m'] = spy.pct_change(252)    # Past 12 months (252 trading days/year)

# Realized volatility (rolling std of daily returns)
feat['rvol_spy_63d'] = spy.pct_change().rolling(63).std()

# Volatility ratio (recent vol / trailing vol = expansion signal)
vol_recent = spy.pct_change().rolling(21).std()
vol_trailing = spy.pct_change().rolling(126).std()
feat['vol_ratio_spy'] = vol_recent / vol_trailing

print('SPY Momentum & Volatility:')
print(f'  ret_spy_1m, ret_spy_3m, ret_spy_6m, ret_spy_12m')
print(f'  rvol_spy_63d (realized volatility)')
print(f'  vol_ratio_spy (recent/trailing vol expansion signal)')
print(f'\nFeatures so far: {feat.shape[1]}')


## Part 1B: Asset Class Signals

Rotation between growth (QQQ), value (XLE), safety (GLD), and inflation (Oil)

In [ ]:
# Load other asset classes
symbols_to_load = {'QQQ': 'qqq', 'XLE': 'xle', 'GLD': 'gld', 'CL=F': 'oil'}
prices = {}

for symbol, name in symbols_to_load.items():
    df = con.execute(f"""
        SELECT date, close FROM silver_macro_prices
        WHERE symbol = '{symbol}' ORDER BY date
    """).df()
    df['date'] = pd.to_datetime(df['date'])
    prices[name] = df.set_index('date')['close'].sort_index()

# QQQ returns (tech leadership)
feat['ret_qqq_3m'] = prices['qqq'].pct_change(63)
feat['ret_qqq_6m'] = prices['qqq'].pct_change(126)

# Spread: QQQ outperformance vs SPY (growth premium)
feat['qqq_spy_spread_6m'] = feat['ret_qqq_6m'] - feat['ret_spy_6m']

# XLE returns (energy/inflation)
feat['ret_xle_3m'] = prices['xle'].pct_change(63)
feat['ret_xle_6m'] = prices['xle'].pct_change(126)

# GLD returns (flight to safety)
feat['ret_gld_3m'] = prices['gld'].pct_change(63)
feat['ret_gld_6m'] = prices['gld'].pct_change(126)

# Oil returns (demand/inflation proxy)
feat['ret_oil_1m'] = prices['oil'].pct_change(21)
feat['ret_oil_3m'] = prices['oil'].pct_change(63)
feat['ret_oil_6m'] = prices['oil'].pct_change(126)

print('Asset Class Signals (Rotation):')
print(f'  QQQ: ret_qqq_3m, ret_qqq_6m, qqq_spy_spread_6m')
print(f'  XLE (Energy): ret_xle_3m, ret_xle_6m')
print(f'  GLD (Safety): ret_gld_3m, ret_gld_6m')
print(f'  Oil: ret_oil_1m, ret_oil_3m, ret_oil_6m')
print(f'\nFeatures so far: {feat.shape[1]}')


## Part 1C: Market Breadth

Health of the rally - what % of S&P 500 is participating?

In [ ]:
# Breadth from gold_panel (S&P 500 constituent data)
breadth_df = con.execute("""
    SELECT date,
           AVG(CASE WHEN ret_252d > 0        THEN 1.0 ELSE 0.0 END) AS breadth_pos_12m,
           AVG(CASE WHEN ret_21d  > 0        THEN 1.0 ELSE 0.0 END) AS breadth_pos_1m,
           AVG(CASE WHEN pct_from_ath > -0.10 THEN 1.0 ELSE 0.0 END) AS breadth_near_ath,
           AVG(CASE WHEN rvol_252d > 0.30    THEN 1.0 ELSE 0.0 END) AS breadth_rvol_elevated
    FROM gold_panel
    WHERE index_type = 'S&P 500'
    GROUP BY date
    ORDER BY date
""").df()

breadth_df['date'] = pd.to_datetime(breadth_df['date'])
breadth = breadth_df.set_index('date')

# Join breadth to features
feat = feat.join(breadth, how='left')

# Forward-fill breadth (monthly indicator, fill daily)
for col in breadth.columns:
    feat[col] = feat[col].ffill()

print('Market Breadth Signals:')
print('  breadth_pos_12m     → % of S&P 500 up over past year')
print('  breadth_pos_1m      → % of S&P 500 up over past month')
print('  breadth_near_ath    → % within 10% of 52-week high')
print('  breadth_rvol_elevated → % with vol > 30%')
print(f'\nFeatures so far: {feat.shape[1]}')


## Part 1D: Fear Gauge

Credit stress, volatility sentiment, yield curve inversion

In [ ]:
# VIX - volatility sentiment
vix_df = con.execute("""
    SELECT date, close FROM silver_macro_prices
    WHERE symbol = 'VIX' ORDER BY date
""").df()

vix_df['date'] = pd.to_datetime(vix_df['date'])
vix = vix_df.set_index('date')['close'].sort_index()

feat['vix_level'] = vix
feat['vix_chg_21d'] = vix.pct_change(21)

# High-yield spreads - credit stress
hy_df = con.execute("""
    SELECT date, close FROM silver_macro_prices
    WHERE symbol = 'HY_OAS' ORDER BY date
""").df()

hy_df['date'] = pd.to_datetime(hy_df['date'])
hy = hy_df.set_index('date')['close'].sort_index()

feat['hy_spread'] = hy
feat['hy_spread_chg_21d'] = hy.pct_change(21)
feat['hy_spread_chg_63d'] = hy.pct_change(63)

# Yield curve spread (10Y - 3M) - recession indicator
y10_df = con.execute("""
    SELECT date, close FROM silver_macro_prices
    WHERE symbol = 'TNX_10Y' ORDER BY date
""").df()

y3m_df = con.execute("""
    SELECT date, close FROM silver_macro_prices
    WHERE symbol = 'IRX_3M' ORDER BY date
""").df()

y10_df['date'] = pd.to_datetime(y10_df['date'])
y3m_df['date'] = pd.to_datetime(y3m_df['date'])

y10 = y10_df.set_index('date')['close'].sort_index()
y3m = y3m_df.set_index('date')['close'].sort_index()

feat['ted_spread'] = y10 - y3m  # Term premium

print('Fear Gauge (Stress Signals):')
print('  vix_level, vix_chg_21d               → Volatility sentiment')
print('  hy_spread, hy_spread_chg_21d, chg_63d → Credit stress')
print('  ted_spread                            → Yield curve / term premium')
print(f'\n✓ Complete feature set: {feat.shape[1]} features')


## Part 2: Build Regime Labels (Ground Truth)

Use forward 63-day SPY return to label regimes

In [ ]:
# Forward return: what the market actually does in next 63 days
feat['fwd_ret_63d'] = spy.pct_change(63).shift(-63)

# Trailing return: market context (past 6 months)
feat['trail_ret_126d'] = spy.pct_change(126)

# Regime classification (production logic)
def forward_regime(row):
    """Classify regime based on forward return + context"""
    fwd = row['fwd_ret_63d']
    trail = row['trail_ret_126d']
    
    if pd.isna(fwd) or pd.isna(trail):
        return np.nan
    
    if fwd > 0.07:
        return 'EXPANSION'
    elif fwd < -0.05:
        return 'CONTRACTION'
    elif fwd > 0.0 and trail < -0.05:
        return 'RECOVERY'
    else:
        return 'LATE_CYCLE'

feat['regime'] = feat.apply(forward_regime, axis=1)

# Drop rows with missing labels
df_labeled = feat.dropna(subset=['regime']).copy()

print('Regime Distribution:')
print(df_labeled['regime'].value_counts())
print(f'\n{len(df_labeled)} labeled observations ready for training')


## Part 3: Train GradientBoostingClassifier

Production hyperparameters exactly as used in regime_ml.py

In [ ]:
# Prepare data for training
X = df_labeled.drop(['fwd_ret_63d', 'trail_ret_126d', 'regime'], axis=1)
y = df_labeled['regime']

# Drop rows with any NaN values (early dates where returns can't be computed)
valid_idx = X.notna().all(axis=1)
X = X[valid_idx]
y = y[valid_idx]

print(f'After dropping NaN: {X.shape[0]} samples × {X.shape[1]} features')
print(f'Date range: {X.index.min()} to {X.index.max()}')

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train GradientBoostingClassifier with PRODUCTION hyperparameters
model = GradientBoostingClassifier(
    n_estimators=300,          # 300 trees
    max_depth=4,               # Shallow trees (prevent overfitting)
    learning_rate=0.05,        # Slow learning (robust)
    subsample=0.8,             # Use 80% of data per iteration
    min_samples_leaf=20,       # Require 20+ samples in leaves
    validation_fraction=0.1,   # 10% validation set
    n_iter_no_change=30,       # Early stopping
    random_state=42
)

print('Training GradientBoostingClassifier...')
model.fit(X, y_encoded)

# Calculate training accuracy
train_acc = model.score(X, y_encoded)

print(f'\n✓ Model trained!')
print(f'  Estimators used: {model.n_estimators_}')
print(f'  Train accuracy: {train_acc:.1%}')


## Part 4: Feature Importance

Which features matter most for regime classification?

In [ ]:
# Get feature importance
importances = model.feature_importances_
feature_names = X.columns.tolist()

# Sort by importance
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

# Plot top 15 features
fig, ax = plt.subplots(figsize=(10, 8))
top_15 = importance_df.head(15)
ax.barh(range(len(top_15)), top_15['importance'].values, color='steelblue')
ax.set_yticks(range(len(top_15)))
ax.set_yticklabels(top_15['feature'].values)
ax.set_xlabel('Importance', fontweight='bold')
ax.set_title('Top 15 Features: regime_ml.py Model', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../outputs/04_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n✓ Feature importance plot saved')
print('\nTop 10 Features:')
print(importance_df.head(10).to_string(index=False))

# Save model and label encoder for Phase 5
os.makedirs('../data', exist_ok=True)
with open('../data/regime_ml_model.pkl', 'wb') as f:
    pickle.dump({'model': model, 'label_encoder': le, 'features': X.columns.tolist()}, f)

print('\n✓ Model saved to data/regime_ml_model.pkl')
print('\nReady for Phase 5: Real-time Inference')
